# Module 3, Class 2 Assignment: Scaling

**Student:** Firdavs Boliev

This notebook completes the Module 3 Class 2 homework in a beginner-friendly way. The code cells include clear comments that explain what each step is doing.

## What this notebook covers
- Compared the raw ranges of tenure, MonthlyCharges, and TotalCharges.
- Applied StandardScaler, MinMaxScaler, and RobustScaler to the same numeric features.
- Split the data before fitting a scaler to avoid data leakage.
- Saved and reloaded the trained scaler with joblib.


In [1]:
# This setup cell imports the libraries, loads the Telco dataset, and prepares TotalCharges as a numeric feature.
# === SETUP — run this first ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
print('Loaded:', df.shape)


Loaded: (7043, 21)


### Cell 1 — see how different the ranges are

In [2]:
# This cell compares the original numeric ranges before scaling.
cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[cols].describe().round(2)


,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80


### Cell 2 — apply StandardScaler (mean=0, std=1)

In [3]:
# This cell applies StandardScaler so each feature has mean 0 and standard deviation 1.
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = df[cols]
X_std = scaler.fit_transform(X)
X_std_df = pd.DataFrame(X_std, columns=cols)
X_std_df.describe().round(2)


,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,-0.00,-0.00,-0.00
std,1.00,1.00,1.00
min,-1.32,-1.55,-1.01
25%,-0.95,-0.97,-0.83
50%,-0.14,0.19,-0.39
75%,0.92,0.83,0.66
max,1.61,1.79,2.83


### Cell 3 — apply MinMaxScaler (range 0 to 1)

In [4]:
# This cell applies MinMaxScaler so each feature is moved into the 0 to 1 range.
from sklearn.preprocessing import MinMaxScaler
mm = MinMaxScaler()
X_mm = mm.fit_transform(X)
X_mm_df = pd.DataFrame(X_mm, columns=cols)
X_mm_df.agg(['min', 'max']).round(2)


,tenure,MonthlyCharges,TotalCharges
min,0.0,0.0,0.0
max,1.0,1.0,1.0


### Cell 4 — apply RobustScaler (uses median + IQR, ignores outliers)

In [5]:
# This cell applies RobustScaler, which uses the median and IQR and is less sensitive to outliers.
from sklearn.preprocessing import RobustScaler
rb = RobustScaler()
X_rb = rb.fit_transform(X)
X_rb_df = pd.DataFrame(X_rb, columns=cols)
X_rb_df.describe().round(2)


,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,0.07,-0.10,0.26
std,0.53,0.55,0.67
min,-0.63,-0.96,-0.41
25%,-0.43,-0.64,-0.29
50%,0.00,0.00,0.00
75%,0.57,0.36,0.71
max,0.93,0.89,2.15


### Cell 5 — split FIRST, then scale (the big rule — no data leakage)

In [6]:
# This cell splits the data first, fits the scaler only on training data, and transforms both train and test data.
from sklearn.model_selection import train_test_split
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f'Train shape: {X_train_s.shape}, Test shape: {X_test_s.shape}')
print(f'Train mean (should be ~0): {X_train_s.mean(axis=0).round(2)}')


Train shape: (5634, 3), Test shape: (1409, 3)
Train mean (should be ~0): [ 0. -0.  0.]


### Cell 6 — save the trained scaler to disk

In [7]:
# This cell saves the fitted scaler to disk and loads it again to confirm the saved object works.
import joblib
joblib.dump(scaler, 'my_scaler.joblib')
loaded = joblib.load('my_scaler.joblib')
print('Saved and loaded. Learned mean:', loaded.mean_.round(2))


Saved and loaded. Learned mean: [  32.37   64.86 2287.09]
